# FGVC8 (Plant Pathology 2021) field evaluation - executed results

This notebook holds the executed FGVC8 section pulled from `ADDC_final_pipeline.ipynb` (run end-to-end on 2026-07-08). Outputs (tables/figures) also copied to `./outputs/`.

It is a **fragment**, not standalone: it depends on variables from the full pipeline (CLASSES, train_df, val_df, train_cls, cls_loader, cls_predict, CLS_MODELS, BEST_CLS, SEED, MODEL_SIZE, NICE, OUT) which only exist after running the pipeline notebook up to the classification + PlantDoc generalization sections. Shown here for reference/results, not for re-execution as-is.

## 7b. FGVC8 (Plant Pathology 2021) - second real-field evaluation set
Zero-shot + 5-fold cross-validation, mirroring the PlantDoc generalization section above.

In [22]:
# %% ---- FGVC8 Cell A: build fgvc8_all_df (filter + map + de-duplicate) ----
import hashlib
from pathlib import Path
import pandas as pd

FGVC8_ROOT   = Path("../FGVC8/plant-pathology-2021-fgvc8")
FGVC8_IMAGES = FGVC8_ROOT / "train_images"
FGVC8_CSV    = FGVC8_ROOT / "train.csv"

# subsample cap per class to keep 5-fold retraining feasible on your GPU.
# Set to None to use every image (heavier but most thorough).
MAX_PER_CLASS = 700   # e.g. 700 -> ~2,800 images total; raise/lower as compute allows

# FGVC8 single-label token -> thesis class string
FGVC8_MAP = {
    "scab":               "Apple___Apple_scab",
    "frog_eye_leaf_spot": "Apple___Black_rot",       # frogeye = black rot (same fungus)
    "rust":               "Apple___Cedar_apple_rust",
    "healthy":            "Apple___healthy",
}

df = pd.read_csv(FGVC8_CSV)
# keep only SINGLE-label rows whose single label is one of the four we want
df["labels"] = df["labels"].astype(str).str.strip()
df = df[df["labels"].isin(FGVC8_MAP.keys())].copy()           # single-token label only
df["class"] = df["labels"].map(FGVC8_MAP)
df["image_path"] = df["image"].apply(lambda n: str(FGVC8_IMAGES / n))
df = df[df["image_path"].apply(lambda p: Path(p).exists())]   # keep files that exist

# de-duplicate by exact file content (removes any repeated images)
def _md5(path, buf=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(buf), b""):
            h.update(chunk)
    return h.hexdigest()

df["_h"] = df["image_path"].apply(_md5)
before = len(df)
df = df.drop_duplicates("_h").drop(columns="_h").reset_index(drop=True)
print(f"FGVC8: kept {len(df)} single-label images "
      f"({before - len(df)} exact duplicates removed).")

# optional balanced subsample per class (reproducible)
if MAX_PER_CLASS is not None:
    df = (df.groupby("class", group_keys=False)
            .apply(lambda g: g.sample(n=min(len(g), MAX_PER_CLASS), random_state=SEED))
            .reset_index(drop=True))

fgvc8_all_df = df[["image_path", "class"]].copy()
print("FGVC8 class counts (field, 4 classes):")
print(fgvc8_all_df["class"].map(NICE).value_counts())
print("Total FGVC8 field images used:", len(fgvc8_all_df))


FGVC8: kept 14491 single-label images (0 exact duplicates removed).
FGVC8 class counts (field, 4 classes):
class
Apple scab          700
Black rot           700
Cedar apple rust    700
healthy             700
Name: count, dtype: int64
Total FGVC8 field images used: 2800


In [23]:
# %% ---- FGVC8 Cell B: zero-shot + 5-fold cross-validation ----------------
#  Mirrors the PlantDoc evaluation exactly, but on FGVC8 and on all FOUR classes.
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1) ZERO-SHOT: PlantVillage-only best model applied to all FGVC8 images
yt, pp = cls_predict(CLS_MODELS[BEST_CLS], cls_loader(fgvc8_all_df, MODEL_SIZE[BEST_CLS], False))
fgvc8_zero = accuracy_score(yt, pp.argmax(1))
print(f"[FGVC8] Zero-shot ({BEST_CLS} trained on PlantVillage only): {fgvc8_zero*100:.2f}%")

# 2) 5-FOLD CV: FGVC8 folds mixed into PlantVillage training, each fold tested once
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
y = fgvc8_all_df["class"].map(CLASSES.index).values
fold_acc, cv_yt, cv_pp = [], [], []
for k, (tr_idx, te_idx) in enumerate(skf.split(fgvc8_all_df, y)):
    f_tr, f_te = fgvc8_all_df.iloc[tr_idx], fgvc8_all_df.iloc[te_idx]
    mixed = pd.concat([train_df, f_tr], ignore_index=True)     # PV + FGVC8 training folds
    m, _ = train_cls(f"{BEST_CLS}_fgvc8_fold{k}", mixed, val_df,
                     CLS_PHASE1, CLS_PHASE2, verbose=False, arch=BEST_CLS)
    ytk, ppk = cls_predict(m, cls_loader(f_te, MODEL_SIZE[BEST_CLS], False))
    a = accuracy_score(ytk, ppk.argmax(1)); fold_acc.append(a)
    cv_yt.append(ytk); cv_pp.append(ppk)
    print(f"[FGVC8] fold {k+1}/5: test acc = {a*100:.1f}%  (n={len(f_te)})")

yt = np.concatenate(cv_yt); pp = np.concatenate(cv_pp)          # each image tested once
present = sorted(np.unique(yt).tolist())
mean, std = float(np.mean(fold_acc)), float(np.std(fold_acc))
print(f"\n[FGVC8] Zero-shot: {fgvc8_zero*100:.2f}%   ->   5-fold CV (FGVC8 mixed in "
      f"training): {mean*100:.2f}% +/- {std*100:.2f}%")
print("Per-fold:", " / ".join(f"{a*100:.1f}" for a in fold_acc))
print(classification_report(yt, pp.argmax(1), labels=present,
                            target_names=[NICE[CLASSES[i]] for i in present], zero_division=0))

# 3) save tables + confusion matrix figure (same style as the PlantDoc outputs)
pd.DataFrame([{"metric": "fgvc8_zero_shot",  "value": round(fgvc8_zero, 4)},
              {"metric": "fgvc8_cv_mean",     "value": round(mean, 4)},
              {"metric": "fgvc8_cv_std",      "value": round(std, 4)},
              *[{"metric": f"fgvc8_fold{i+1}", "value": round(a, 4)} for i, a in enumerate(fold_acc)]]
             ).to_csv(OUT / "tables/generalization_fgvc8.csv", index=False)

import matplotlib.pyplot as plt
cm = confusion_matrix(yt, pp.argmax(1), labels=present)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(cm, cmap="Blues"); fig.colorbar(im, ax=ax)
names = [NICE[CLASSES[i]] for i in present]
ax.set_xticks(range(len(present))); ax.set_xticklabels(names, rotation=45, ha="right")
ax.set_yticks(range(len(present))); ax.set_yticklabels(names)
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title("FGVC8 5-fold CV confusion matrix")
for i in range(len(present)):
    for j in range(len(present)):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.tight_layout(); fig.savefig(OUT / "figures/fgvc8_confusion.png", dpi=150); plt.close(fig)
print("Saved: outputs/tables/generalization_fgvc8.csv  and  outputs/figures/fgvc8_confusion.png")


[FGVC8] Zero-shot (resnet50 trained on PlantVillage only): 32.86%


[FGVC8] fold 1/5: test acc = 92.7%  (n=560)


[FGVC8] fold 2/5: test acc = 96.2%  (n=560)


[FGVC8] fold 3/5: test acc = 94.5%  (n=560)


[FGVC8] fold 4/5: test acc = 93.9%  (n=560)


[FGVC8] fold 5/5: test acc = 93.0%  (n=560)

[FGVC8] Zero-shot: 32.86%   ->   5-fold CV (FGVC8 mixed in training): 94.07% +/- 1.26%
Per-fold: 92.7 / 96.2 / 94.5 / 93.9 / 93.0
                  precision    recall  f1-score   support

      Apple scab       0.90      0.92      0.91       700
       Black rot       0.94      0.96      0.95       700
Cedar apple rust       0.99      0.95      0.97       700
         healthy       0.93      0.94      0.94       700

        accuracy                           0.94      2800
       macro avg       0.94      0.94      0.94      2800
    weighted avg       0.94      0.94      0.94      2800

Saved: outputs/tables/generalization_fgvc8.csv  and  outputs/figures/fgvc8_confusion.png
